# Распознавание по вашему фото или видео
Откройте этот файл в **Google Colab** и выполните ячейки сверху вниз. Это **готовые веса**, заново обучать модель не нужно. Выберите файл в третьей ячейке. Исходник останется без изменений; копия с рамками и текстовые координаты сохранятся в отдельную папку на вашем Диске.

`ground_bump` — деревянная кочка **на полу**, `speed_bump` — **знак** неровности. `cone` — оранжево-белый конус/столбик полигона. `traffic_light_red/yellow/green` — предполагаемый цвет; просто `traffic_light` — сигнал не определён. Это просмотр предсказаний, не управление роботом.

In [ ]:
%pip -q install ultralytics==8.4.215

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from ultralytics import YOLO

# Измените путь, если папка yolo лежит не в корне «Моего диска».
folder = Path('/content/drive/MyDrive/yolo')
result_folder = folder / 'Результат YOLO 2026-11-18'
weights = result_folder / 'yolo-drive.pt'
source = folder / 'VID_20261118_124149_236.mp4'  # можно подставить свой JPG или MP4
if not weights.is_file() or not source.is_file():
    raise FileNotFoundError(f'Проверьте пути: {weights} и {source}')

model = YOLO(str(weights))
run = datetime.now(timezone.utc).strftime('run-%Y%m%d-%H%M%S-%f')
frames = 0
for result in model.predict(source=str(source), imgsz=416, conf=0.25, iou=0.7,
                            project=str(result_folder / 'Предсказания'), name=run,
                            stream=True, save=True, save_txt=True, save_conf=True, verbose=False):
    frames += 1
print('Обработано кадров:', frames)
print('Файл с рамками и папка labels:', result_folder / 'Предсказания' / run)